# Adversarial Dataset Generation

**Goal:** Combine verified stealthy adversarial examples with clean training pairs to produce the defense training dataset.

The generator selects only the adversarial examples that:
1. Were generated by the BERT MLM attack.
2. Passed the dual SBERT + GPT-2 stealth filter (semantically coherent and fluent).
3. Caused a measurable reward drop (reward_drop > 0.001).

These are mixed with clean chosen/rejected pairs at a 25% adversarial ratio.

In [ ]:
import sys
import os
import json
import pandas as pd
import matplotlib.pyplot as plt

sys.path.insert(0, '..')

plt.rcParams.update({
    "font.family": "DejaVu Sans",
    "font.size": 10,
    "axes.titlesize": 13,
    "axes.labelsize": 11,
    "figure.dpi": 150,
})

### 1. Generate the Defense Dataset

In [ ]:
from src.robustness.adv_dataset_generator import generate_adversarial_dataset

stats = generate_adversarial_dataset(
    clean_data_path=os.path.join('..', 'data', 'test_data.json'),
    eval_csv_path=os.path.join('..', 'results', 'attack_eval_results.csv'),
    output_path=os.path.join('..', 'data', 'adv_training_pairs.json'),
    adv_ratio=0.25,
    attack_type='BERT MLM'
)

### 2. Inspect the Dataset

In [ ]:
with open(os.path.join('..', 'data', 'adv_training_pairs.json'), 'r') as f:
    pairs = json.load(f)

df = pd.DataFrame(pairs)
print(f"Total pairs: {len(df)}")
print(f"\nComposition:")
print(df['is_adversarial'].value_counts().rename({True: 'Adversarial', False: 'Clean'}))
df.head()

### 3. Dataset Composition

The 25% adversarial mixing ratio is the starting point for the ablation study. During defense training, we will train three separate models at 10%, 25%, and 50% adversarial ratio to find the optimal trade-off between clean accuracy and adversarial robustness.

In [ ]:
labels = ['Clean Pairs', 'Adversarial Pairs']
sizes = [stats['clean_pairs'], stats['adversarial_pairs']]
colors = ['#C39BD3', '#F1948A']

fig, ax = plt.subplots(figsize=(6, 6))
ax.pie(sizes, labels=labels, colors=colors, autopct='%1.1f%%',
       startangle=90, wedgeprops={'edgecolor': 'white', 'linewidth': 2})
ax.set_title('Defense Dataset Composition', pad=12)
plt.tight_layout()

fig_path = os.path.join('..', 'results', '07(1)_dataset_composition.png')
plt.savefig(fig_path, dpi=300, bbox_inches='tight')
print(f"Saved to {fig_path}")
plt.show()